# DGP tests

In [41]:
import numpy as np
import plotly.figure_factory as ff
import plotly.graph_objects as go
import plotly.express as px
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.dgp import generate_covariates, f_alpha, f_linear, f_nonlinear, g_star, propensity_score, generate_treatment, generate_dataset
from src.utils import load_config

## Covariate check 

In [42]:
X = generate_covariates(10000, 20)
X1 = X[:, 0]

fig = ff.create_distplot([X1], ["X1"])
fig.show()

## F linear and non linear check

In [43]:
config = load_config("baseline")

In [44]:
f_lin = f_linear(X, config["linear_weights"])
f_nonlin = f_nonlinear(X, config["nonlinear_terms"])


print("Linear mean:", f_lin.mean(), "std:", f_lin.std())
print("Nonlinear mean:", f_nonlin.mean(), "std:", f_nonlin.std())

Linear mean: 0.004998027230659694 std: 1.1985466796042707
Nonlinear mean: 0.01568817825182928 std: 1.3611435400375629


In [45]:
print(np.corrcoef(f_lin, f_nonlin)[0, 1])

0.04304931939753827


In [46]:
f_a = f_alpha(X, 0.5, config["linear_weights"], config["nonlinear_terms"])

print("Linear mean:", f_a.mean(), "std:", f_a.std())

Linear mean: 0.010343102741244482 std: 0.9259706648396339


## Propensity score

In [47]:
g = g_star(X, 0.5)
print(g.mean(), g.std())

1.1546319456101627e-18 1.0


In [48]:
for k in [0.5, 1.5, 3]:
    for alpha_d in [0, 0.5, 1]:
        e = propensity_score(X, alpha_d=alpha_d, k=k)
        print(f"k={k}, alpha_d={alpha_d}, overlap={np.mean(e*(1-e)):.3f}")

k=0.5, alpha_d=0, overlap=0.236
k=0.5, alpha_d=0.5, overlap=0.236
k=0.5, alpha_d=1, overlap=0.237
k=1.5, alpha_d=0, overlap=0.177
k=1.5, alpha_d=0.5, overlap=0.172
k=1.5, alpha_d=1, overlap=0.182
k=3, alpha_d=0, overlap=0.116
k=3, alpha_d=0.5, overlap=0.106
k=3, alpha_d=1, overlap=0.119


## Generating treatment 

In [49]:
D0, e0 = generate_treatment(X, alpha_d=0, k=1, c=0, seed=42)
print(D0.mean())
print(np.corrcoef(D0, X[:, 0])[0, 1])

0.5019
0.4220787720105174


In [50]:
D1, e1 = generate_treatment(X, alpha_d=1, k=1, c=0, seed=42)
print(D1.mean())
print(np.corrcoef(D1, X[:, 0])[0, 1])

0.4842
0.28260096595117534


## Generating dataset

In [51]:
data = generate_dataset(config, alpha_y=0.5, alpha_d=0.5, kappa=1, seed=42)

print(data["X"].shape)
print(data["D"].mean())
print(data["Y"].mean(), data["Y"].std())
print(data["tau_true"])

(500, 20)
0.514
1.2408802230640297 1.8939740901921427
2.5


In [52]:
print(data["D"][:10])
print(data["Y"][:10])

[1 1 0 0 1 1 1 1 1 0]
[ 1.88658174  2.22890996 -0.54556696  1.34444779  2.99622322  0.44104723
  1.65230547  2.46136466  2.32021667 -1.13148049]


## Sanity plots of dataset

In [53]:
fig = ff.create_distplot(
    [data["X"][:, 0], data["X"][:, 1]],
    ["X1", "X2"],
)
fig.update_layout(
    title={
        "text": "Distribution of X1 and X2",
        "x": 0.5,
        "xanchor": "center"
    }
)
fig.show()

In [54]:
data_0 = generate_dataset(config, alpha_y=0.5, alpha_d=0,kappa=1 ,seed=42)
data_1 = generate_dataset(config, alpha_y=1, alpha_d=1,kappa=1,  seed=42)


In [55]:
fig = ff.create_distplot(
    [data_1["e"], data_0["e"]],
    ["alpha_d = 1", "alpha_d = 0"],
    show_curve=False,
    show_hist=True,
    show_rug=True,
    bin_size=0.02,
    colors=["#1f77b4", "#d62728"],
)
fig.add_vline(
    x=np.mean(data_0["e"]),
    line_width=3,
    line_dash="dot",
    line_color="#d62728",
    annotation_text="alpha_d = 0",
    annotation_position="top left",
)

fig.update_layout(
    title={
        "text": "Distribution of Propensity scores",
        "x": 0.5,
        "xanchor": "center"
    },
    legend_title_text="Group",
)
fig.show()

In [56]:
fig = ff.create_distplot(
    [data_1["Y"], data_0["Y"]],
    ["alpha_d = 1", "alpha_d = 0"],
    show_curve=False,
    show_hist=True,
    show_rug=True,
    colors=["#1f77b4", "#d62728"],
)

fig.update_layout(
    title={
        "text": "Distribution of Outcome",
        "x": 0.5,
        "xanchor": "center"
    },
    legend_title_text="Group",
)
fig.show()

In [57]:
p0 = data_0["D"].mean()
p1 = data_1["D"].mean()

fig = go.Figure()

fig.add_trace(go.Bar(
    name="alpha_d = 0",
    x=["Control (D=0)", "Treated (D=1)"],
    y=[1 - p0, p0],
    marker_color="#d62728"
))

fig.add_trace(go.Bar(
    name="alpha_d = 1",
    x=["Control (D=0)", "Treated (D=1)"],
    y=[1 - p1, p1],
    marker_color="#1f77b4"
))

fig.update_layout(
    barmode="group",
    title={"text": "Treatment Assignment Shares", "x": 0.5},
    yaxis_title="Share of observations",
    xaxis_title="Treatment status",
    legend_title_text="Group"
)

fig.show()

In [58]:
mask_control = data_1["D"] == 0
mask_treated = data_1["D"] == 1

x1_control = data_1["X"][mask_control, 0]
x1_treated = data_1["X"][mask_treated, 0]


fig = go.Figure()

fig.add_trace(go.Box(
    y=x1_control,
    name="Control (D=0)",
    marker_color="#d62728",
    boxmean=True
))

fig.add_trace(go.Box(
    y=x1_treated,
    name="Treated (D=1)",
    marker_color="#1f77b4",
    boxmean=True
))

fig.update_layout(
    title={"text": "Distribution of X1 by Treatment Group", "x": 0.5},
    yaxis_title="X1",
    xaxis_title="Treatment group"
)

fig.show()

In [59]:
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=f_lin,
    name="f_linear",
    opacity=0.55,
    nbinsx=40,
    marker_color="#1f77b4",
    histnorm="probability density"
))

fig.add_trace(go.Histogram(
    x=f_nonlin,
    name="f_nonlinear",
    opacity=0.55,
    nbinsx=40,
    marker_color="#ff7f0e",
    histnorm="probability density"
))

fig.add_trace(go.Histogram(
    x=f_a,
    name="f_alpha (alpha_y = 0.5)",
    opacity=0.55,
    nbinsx=40,
    marker_color="#2ca02c",
    histnorm="probability density"
))

fig.update_layout(
    barmode="overlay",
    title={"text": "Outcome Signal Components", "x": 0.5},
    xaxis_title="Signal value",
    yaxis_title="Density",
    legend_title_text="Component"
)

fig.show()


In [60]:
corr = np.corrcoef(X, rowvar=False)

fig = go.Figure(data=go.Heatmap(
    z=corr,
    colorscale="RdBu",
    zmid=0,
    colorbar_title="Correlation"
))

fig.update_layout(
    title={"text": "Covariate Correlation Matrix", "x": 0.5},
    xaxis_title="Covariate index",
    yaxis_title="Covariate index"
)

fig.show()